In [1]:
from pathlib import Path
import time
import numpy as np
import pandas as pd

from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.base import clone

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

RANDOM_STATE = 42

BALANCE_METHODS = {
    "NONE": None,
    "SMOTE": SMOTE(random_state=RANDOM_STATE),
    "UNDER": RandomUnderSampler(random_state=RANDOM_STATE),
    "SMOTEENN": SMOTEENN(random_state=RANDOM_STATE),
}


def latest_dir(base: Path) -> Path:
    if not base.exists():
        raise FileNotFoundError(f"No existe: {base}")
    subs = [p for p in base.iterdir() if p.is_dir()]
    if not subs:
        raise FileNotFoundError(f"No hay subcarpetas en: {base}")
    return sorted(subs)[-1]


def find_latest_report(report_root: Path, pattern: str) -> tuple[Path, Path]:
    files = sorted(report_root.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No se encontro archivo con patron: {pattern}")
    csv_path = files[-1]
    return csv_path.parent, csv_path


def choose_best_general(df: pd.DataFrame) -> pd.Series:
    req = {"modelo", "balanceo", "cv_macro_f1", "cv_recall_target"}
    miss = req.difference(df.columns)
    if miss:
        raise ValueError(f"Faltan columnas requeridas: {miss}")
    ordered = df.sort_values(["cv_macro_f1", "cv_recall_target"], ascending=[False, False])
    return ordered.iloc[0]


def update_row_inplace(df: pd.DataFrame, row_mask: pd.Series, metrics: dict):
    for k, v in metrics.items():
        if k not in df.columns:
            df[k] = np.nan
        df.loc[row_mask, k] = v


def classes_from_report(report_df: pd.DataFrame):
    excluded = {"accuracy", "macro avg", "weighted avg"}
    out = []
    for idx in report_df.index:
        if str(idx) not in excluded:
            out.append(str(idx))
    return out


def save_eval_pdf(y_test, y_pred_test, y_proba, class_order, pdf_path: Path, title_prefix: str):
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    cm = confusion_matrix(y_test, y_pred_test, labels=class_order)

    with PdfPages(pdf_path) as pdf:
        ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_order).plot(cmap="Blues")
        plt.title(f"{title_prefix} - Matriz de Confusion")
        pdf.savefig(); plt.close()

        unique_classes = np.unique(y_test)
        if unique_classes.size >= 2:
            y_bin = label_binarize(y_test, classes=class_order)
            proba_for_curves = y_proba

            if y_bin.shape[1] == 1:
                y_bin = np.hstack([1 - y_bin, y_bin])
                if proba_for_curves.shape[1] == 1:
                    proba_for_curves = np.hstack([1 - proba_for_curves, proba_for_curves])

            plt.figure()
            for i, clase in enumerate(class_order):
                fpr, tpr, _ = roc_curve(y_bin[:, i], proba_for_curves[:, i])
                roc_auc = auc(fpr, tpr)
                plt.plot(fpr, tpr, label=f"Clase {clase} (AUC={roc_auc:.2f})")
            plt.plot([0, 1], [0, 1], "k--")
            plt.title(f"{title_prefix} - Curvas ROC por clase")
            plt.xlabel("FPR")
            plt.ylabel("TPR")
            plt.legend()
            pdf.savefig(); plt.close()

            plt.figure()
            for i, clase in enumerate(class_order):
                precision, recall, _ = precision_recall_curve(y_bin[:, i], proba_for_curves[:, i])
                pr_auc = average_precision_score(y_bin[:, i], proba_for_curves[:, i])
                plt.plot(recall, precision, label=f"Clase {clase} (PR AUC={pr_auc:.2f})")
            plt.title(f"{title_prefix} - Curvas Precision-Recall por clase")
            plt.xlabel("Recall")
            plt.ylabel("Precision")
            plt.legend()
            pdf.savefig(); plt.close()


In [2]:
from xgboost import XGBClassifier


def build_model(balance_name: str):
    balancer = BALANCE_METHODS[balance_name]
    steps = []
    if balancer is not None:
        steps.append(("balance", clone(balancer)))
    steps.append(("model", XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )))
    return Pipeline(steps)


def train_predict(best_row: pd.Series, input_path: Path, report_dir: Path):
    base_name = str(best_row["modelo"])
    balance_name = str(best_row["balanceo"]).upper()

    X_train = pd.read_parquet(input_path / f"X_train_{base_name}.parquet")
    X_test = pd.read_parquet(input_path / f"X_test_{base_name}.parquet")
    y_train = pd.read_parquet(input_path / f"y_train_{base_name}.parquet").squeeze()
    y_test = pd.read_parquet(input_path / f"y_test_{base_name}.parquet").squeeze()

    y_min = y_train.min()
    y_train_adj = y_train - y_min

    model = build_model(balance_name)
    t0 = time.time()
    model.fit(X_train, y_train_adj)
    tiempo_min = (time.time() - t0) / 60

    y_pred_train = model.predict(X_train) + y_min
    y_pred_test = model.predict(X_test) + y_min
    y_proba = model.predict_proba(X_test)
    class_order = model.named_steps["model"].classes_ + y_min

    pdf_path = report_dir / balance_name / f"reporte_{base_name}_BEST_GENERAL.pdf"
    save_eval_pdf(y_test, y_pred_test, y_proba, class_order, pdf_path, title_prefix=f"{base_name} [{balance_name}]")

    report_train = pd.DataFrame(classification_report(y_train, y_pred_train, output_dict=True, zero_division=0)).T
    report_test = pd.DataFrame(classification_report(y_test, y_pred_test, output_dict=True, zero_division=0)).T

    metrics = {
        "nan_total_train": int(X_train.isna().sum().sum()),
        "nan_total_test": int(X_test.isna().sum().sum()),
        "accuracy_test": float(accuracy_score(y_test, y_pred_test)),
        "macro_f1_test": float(f1_score(y_test, y_pred_test, average="macro", zero_division=0)),
        "weighted_f1_test": float(f1_score(y_test, y_pred_test, average="weighted", zero_division=0)),
        "accuracy_train": float(accuracy_score(y_train, y_pred_train)),
        "macro_f1_train": float(f1_score(y_train, y_pred_train, average="macro", zero_division=0)),
        "weighted_f1_train": float(f1_score(y_train, y_pred_train, average="weighted", zero_division=0)),
        "tiempo_min": float(tiempo_min),
    }
    metrics["sobreajuste_macro_f1"] = metrics["macro_f1_train"] - metrics["macro_f1_test"]

    for c in classes_from_report(report_test):
        metrics[f"f1_{c}_test"] = float(report_test.loc[c, "f1-score"])
        metrics[f"recall_{c}_test"] = float(report_test.loc[c, "recall"])
        metrics[f"precision_{c}_test"] = float(report_test.loc[c, "precision"])
        metrics[f"f1_{c}_train"] = float(report_train.loc[c, "f1-score"])
        metrics[f"recall_{c}_train"] = float(report_train.loc[c, "recall"])
        metrics[f"precision_{c}_train"] = float(report_train.loc[c, "precision"])

    return base_name, balance_name, metrics


In [3]:
report_dir, summary_csv = find_latest_report(
    Path("reports/08_modelo_xgboost"),
    "*/XGBoost_*/resumen_modelos_xgboost.csv",
)
input_path = latest_dir(Path("data/intermediate/05_seleccion"))

print(f"Reporte: {summary_csv}")
print(f"Input:   {input_path}")

df = pd.read_csv(summary_csv)
best = choose_best_general(df)

print("Mejor general segun CV:")
print(best[["modelo", "balanceo", "cv_macro_f1", "cv_recall_target"]].to_dict())

base_name, balance_name, new_metrics = train_predict(best, input_path, report_dir)

new_metrics["cv_macro_f1"] = float(best["cv_macro_f1"])
new_metrics["cv_recall_target"] = float(best["cv_recall_target"])

mask = (df["modelo"].astype(str) == base_name) & (df["balanceo"].astype(str).str.upper() == balance_name)
if mask.any():
    update_row_inplace(df, mask, new_metrics)
    print(f"Fila actualizada en resumen: modelo={base_name}, balanceo={balance_name}")
else:
    row = {"modelo": base_name, "balanceo": balance_name}
    row.update(new_metrics)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    print(f"No existia fila; se agrego nueva para modelo={base_name}, balanceo={balance_name}")

df.to_csv(summary_csv, index=False)
print(f"Resumen actualizado: {summary_csv}")

per_class_cols = [
    c for c in df.columns
    if c.startswith("f1_") or c.startswith("recall_") or c.startswith("precision_")
]
aux_cols = ["modelo", "balanceo", "cv_macro_f1", "cv_recall_target"] + per_class_cols
aux_cols = [c for c in aux_cols if c in df.columns]
aux_df = df.loc[mask, aux_cols].copy() if mask.any() else df.tail(1)[aux_cols].copy()
aux_path = report_dir / "best_general_per_class_metrics_xgboost.csv"
aux_df.to_csv(aux_path, index=False)
print(f"Auxiliar por clase: {aux_path}")
print(f"PDF generado: {report_dir / balance_name / f'reporte_{base_name}_BEST_GENERAL.pdf'}")


Reporte: reports\08_modelo_xgboost\20260210\XGBoost_04_2026-02-10\resumen_modelos_xgboost.csv
Input:   data\intermediate\05_seleccion\04_2026_01_12
Mejor general segun CV:
{'modelo': 'MinMax_FE_PCAopt', 'balanceo': 'SMOTE', 'cv_macro_f1': 0.4202280014168591, 'cv_recall_target': 0.2439882415305286}


c:\Users\Gonzalo\Documents\GitHub\TesisAustral\tesis_austral_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [07:18:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Fila actualizada en resumen: modelo=MinMax_FE_PCAopt, balanceo=SMOTE
Resumen actualizado: reports\08_modelo_xgboost\20260210\XGBoost_04_2026-02-10\resumen_modelos_xgboost.csv
Auxiliar por clase: reports\08_modelo_xgboost\20260210\XGBoost_04_2026-02-10\best_general_per_class_metrics_xgboost.csv
PDF generado: reports\08_modelo_xgboost\20260210\XGBoost_04_2026-02-10\SMOTE\reporte_MinMax_FE_PCAopt_BEST_GENERAL.pdf
